# U23AI035 - LAB 12 - IMAES
## Experiment 12: Expert System Application — A Financial Advisor with Scoring Rules and Hard Constraints

**SVNIT, Surat | Department of Artificial Intelligence | Semester VII**

**Objective:** Build an explainable financial-advisor expert system using (1) a weighted risk-scoring layer, (2) hard constraints that override the score, and (3) Monte Carlo simulation to evaluate recommended and unconstrained allocations.

> **Academic-use note:** The scoring weights, thresholds, example clients, and return assumptions below are transparent illustrative design choices for this lab. They are not individualized financial advice or market forecasts. Run the notebook from top to bottom to reproduce the tables and charts.

## 1. Design choices and assumptions

### 1.1 Risk score
The score is the sum of five components. A larger score indicates greater capacity/willingness to take investment risk.

| Client attribute | Rule | Points |
|---|---|---:|
| Age | Under 30 / 30–44 / 45–59 / 60+ | 20 / 15 / 8 / 0 |
| Investment horizon | 10+ years / 5–9 years / 3–4 years / under 3 years | 25 / 18 / 10 / 0 |
| Income stability | Stable / Variable | 15 / 5 |
| Dependants | 0 / 1–2 / 3+ | 10 / 5 / 0 |
| Stated loss tolerance | High / Moderate / Low | 30 / 18 / 5 |

The maximum score is 100. Score bands map to strategic allocations:

| Score | Band | Equity | Bonds | Cash |
|---:|---|---:|---:|---:|
| 0–24 | Conservative | 20% | 50% | 30% |
| 25–49 | Cautious | 40% | 45% | 15% |
| 50–74 | Balanced | 60% | 30% | 10% |
| 75–100 | Growth | 80% | 15% | 5% |

### 1.2 Hard constraints (applied after scoring)
1. **High-interest debt:** if present, direct the entire investable amount to debt repayment (0% equity, 0% bonds, 100% cash/debt repayment). The simulation does not model the debt balance or interest avoided; this is a decision-priority representation.
2. **Emergency reserve:** if the reserve is inadequate, cap equity at 20% and move the removed amount to cash.
3. **Near-term goal:** if the goal is less than three years away, cap equity at 0% and move the removed amount to cash.

Constraints are applied in the order shown. Each action is recorded in an explanation trace. The high-interest-debt rule is an intentionally strong override, so it supersedes the normal investment allocation.

### 1.3 Simulation assumptions
Annual nominal returns are independently drawn from normal distributions:

| Asset class | Mean annual return | Standard deviation |
|---|---:|---:|
| Equity | 8.0% | 18.0% |
| Bonds | 5.0% | 7.0% |
| Cash | 3.0% | 1.5% |

We simulate 10,000 paths per client over the client's stated horizon. Each year's portfolio return is the weighted sum of the three asset returns, and portfolio value compounds annually. The same random return paths are used to compare constrained and unconstrained allocations (paired comparison).

## 2. Implementation: scoring, bands, constraints and explanation trace

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AGE_POINTS = [(30, 20), (45, 15), (60, 8), (10**9, 0)]
HORIZON_POINTS = [(10, 25), (5, 18), (3, 10), (0, 0)]

BANDS = [
    (0, 24, "Conservative", {"Equity": 0.20, "Bonds": 0.50, "Cash": 0.30}),
    (25, 49, "Cautious",     {"Equity": 0.40, "Bonds": 0.45, "Cash": 0.15}),
    (50, 74, "Balanced",     {"Equity": 0.60, "Bonds": 0.30, "Cash": 0.10}),
    (75, 100, "Growth",      {"Equity": 0.80, "Bonds": 0.15, "Cash": 0.05}),
]

def age_points(age):
    if age < 30: return 20
    if age < 45: return 15
    if age < 60: return 8
    return 0

def horizon_points(years):
    if years >= 10: return 25
    if years >= 5: return 18
    if years >= 3: return 10
    return 0

def risk_score(client):
    parts = {
        "Age": age_points(client["age"]),
        "Horizon": horizon_points(client["horizon"]),
        "Income stability": 15 if client["income_stable"] else 5,
        "Dependants": 10 if client["dependants"] == 0 else (5 if client["dependants"] <= 2 else 0),
        "Loss tolerance": {"High": 30, "Moderate": 18, "Low": 5}[client["loss_tolerance"]],
    }
    return sum(parts.values()), parts

def score_band(score):
    for low, high, name, allocation in BANDS:
        if low <= score <= high:
            return name, allocation.copy()
    raise ValueError("Score outside defined range")

def normalize_allocation(allocation):
    total = sum(allocation.values())
    return {k: v / total for k, v in allocation.items()}

def apply_constraints(client, base_allocation):
    allocation = base_allocation.copy()
    trace = []
    fired = []

    if client["high_interest_debt"]:
        allocation = {"Equity": 0.0, "Bonds": 0.0, "Cash": 1.0}
        trace.append("High-interest debt present: prioritize repayment; no risky investment allocation.")
        fired.append("Debt")

    if not client["emergency_fund_adequate"] and not client["high_interest_debt"]:
        removed = max(0.0, allocation["Equity"] - 0.20)
        allocation["Equity"] -= removed
        allocation["Cash"] += removed
        trace.append(f"Emergency reserve inadequate: equity capped at 20% (shifted {removed:.0%} to cash).")
        fired.append("Emergency reserve")

    if client["horizon"] < 3 and not client["high_interest_debt"]:
        removed = allocation["Equity"]
        allocation["Equity"] = 0.0
        allocation["Cash"] += removed
        trace.append(f"Goal is under 3 years away: equity capped at 0% (shifted {removed:.0%} to cash).")
        fired.append("Near-term goal")

    return normalize_allocation(allocation), trace, fired

def advise(client):
    score, components = risk_score(client)
    band, base = score_band(score)
    final, trace, fired = apply_constraints(client, base)
    if not trace:
        trace = ["No hard constraint triggered; score-based allocation retained."]
    return {
        "score": score, "components": components, "band": band,
        "unconstrained": base, "recommended": final,
        "trace": trace, "constraints_fired": fired
    }

def allocation_string(a):
    return " | ".join(f"{k}: {v:.0%}" for k, v in a.items())

## 3. Three contrasting example clients

In [ ]:
clients = [
    {
        "name": "Asha – Early-career planner", "age": 25, "horizon": 15,
        "income_stable": True, "dependants": 0, "loss_tolerance": "High",
        "high_interest_debt": False, "emergency_fund_adequate": True,
        "initial_amount": 200000
    },
    {
        "name": "Bharat – Family with limited reserve", "age": 38, "horizon": 8,
        "income_stable": True, "dependants": 2, "loss_tolerance": "Moderate",
        "high_interest_debt": False, "emergency_fund_adequate": False,
        "initial_amount": 300000
    },
    {
        "name": "Charu – Near-term goal and debt", "age": 52, "horizon": 2,
        "income_stable": False, "dependants": 3, "loss_tolerance": "Low",
        "high_interest_debt": True, "emergency_fund_adequate": False,
        "initial_amount": 150000
    },
]

advice = {c["name"]: advise(c) for c in clients}
for client in clients:
    result = advice[client["name"]]
    print(f"\n{client['name']}")
    print(f"Risk score: {result['score']}/100 | Band: {result['band']}")
    print("Score components:", result["components"])
    print("Score-only allocation:", allocation_string(result["unconstrained"]))
    print("Final allocation:      ", allocation_string(result["recommended"]))
    print("Explanation trace:")
    for item in result["trace"]:
        print("  •", item)

## 4. Monte Carlo evaluation

In [ ]:
ASSETS = ["Equity", "Bonds", "Cash"]
RETURN_MEAN = np.array([0.08, 0.05, 0.03])
RETURN_STD  = np.array([0.18, 0.07, 0.015])
N_PATHS = 10_000
RANDOM_SEED = 401

def simulate_asset_returns(horizon, n_paths=N_PATHS, seed=RANDOM_SEED):
    # A fixed seed per client makes the experiment reproducible.
    rng = np.random.default_rng(seed)
    return rng.normal(
        loc=RETURN_MEAN, scale=RETURN_STD,
        size=(n_paths, horizon, len(ASSETS))
    )

def simulate_allocation(initial_amount, allocation, asset_returns):
    weights = np.array([allocation[a] for a in ASSETS])
    portfolio_returns = np.sum(asset_returns * weights[None, None, :], axis=2)
    # A normal draw can theoretically be below -100%; floor at -100% to prevent
    # a mathematically impossible negative asset value.
    portfolio_returns = np.maximum(portfolio_returns, -1.0)
    final_values = initial_amount * np.prod(1.0 + portfolio_returns, axis=1)
    return final_values

def summarize_outcomes(values, initial_amount):
    return {
        "Median final value (₹)": np.median(values),
        "5th percentile (₹)": np.percentile(values, 5),
        "Probability of loss (%)": 100 * np.mean(values < initial_amount)
    }

rows = []
simulation_values = {}
for idx, client in enumerate(clients):
    result = advice[client["name"]]
    paths = simulate_asset_returns(client["horizon"], seed=RANDOM_SEED + idx)
    for version, allocation in [
        ("Recommended (constraints applied)", result["recommended"]),
        ("Score-only (no constraints)", result["unconstrained"])
    ]:
        values = simulate_allocation(client["initial_amount"], allocation, paths)
        simulation_values[(client["name"], version)] = values
        summary = summarize_outcomes(values, client["initial_amount"])
        rows.append({
            "Client": client["name"], "Version": version,
            "Allocation": allocation_string(allocation), **summary
        })

simulation_table = pd.DataFrame(rows)
display(simulation_table.style.format({
    "Median final value (₹)": "₹{:,.0f}",
    "5th percentile (₹)": "₹{:,.0f}",
    "Probability of loss (%)": "{:.2f}%"
}))

### 4.1 Visual comparison of outcome distributions
The box plots show the simulated distribution of final portfolio values for each client under the two allocation versions. The fifth percentile is a downside measure; it complements rather than replaces the median.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, client in zip(axes, clients):
    name = client["name"]
    vals = [
        simulation_values[(name, "Recommended (constraints applied)")],
        simulation_values[(name, "Score-only (no constraints)")]
    ]
    ax.boxplot(vals, tick_labels=["With constraints", "Score only"], showfliers=False)
    ax.axhline(client["initial_amount"], linestyle="--", linewidth=1, label="Initial amount")
    ax.set_title(name)
    ax.set_ylabel("Final value (₹)")
    ax.tick_params(axis="x", rotation=12)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Observations and comments

1. **Which client was changed most by the constraint layer?** Charu is changed most: the score-only allocation is replaced by 100% cash/debt repayment because high-interest debt is present. Bharat's equity allocation is also reduced by the emergency-reserve rule. Asha's allocation is unchanged because none of the hard constraints fires.
2. **What would the score alone advise?** It would map each score to its standard band allocation, without considering debt, emergency-fund adequacy, or the short time to a goal. This is why the score-only allocation is shown separately in the output.
3. **What does the fifth percentile add beyond the median?** The median describes the middle outcome, while the fifth percentile estimates a poor-tail outcome: approximately 5% of simulated outcomes fall below it. It helps reveal downside exposure that a median can hide.
4. **Would the explanation trace satisfy a client asking why they must hold cash?** It provides an auditable reason and quantifies the equity amount shifted to cash. A production advisor should add client-specific amounts, explain the emergency-reserve target, and communicate that this is a rule-based priority rather than a guaranteed outcome.
5. **What does the simulation not capture about repaying debt first?** It treats the debt rule as an allocation override; it does not model debt principal, interest rate, repayment schedule, fees, tax effects, or the avoided interest as a comparatively certain benefit. A complete comparison would simulate the debt balance and compare net worth after repayment versus investing.

## 6. Exercise 1 — Population comparison and constraint impact

Generate a synthetic population, compare recommendations with and without constraints, count changed recommendations, and compute the mean fifth-percentile outcome among clients affected by each constraint. For comparability, each simulated client is evaluated with the same return assumptions and number of paths. The synthetic population is a lab stress-test, not a representative survey.

In [ ]:
def generate_population(n=250, seed=2026):
    rng = np.random.default_rng(seed)
    population = []
    for i in range(n):
        age = int(rng.integers(21, 71))
        horizon = int(rng.choice([1, 2, 3, 5, 7, 10, 15, 20]))
        population.append({
            "name": f"Client_{i+1:03d}",
            "age": age,
            "horizon": horizon,
            "income_stable": bool(rng.choice([True, False], p=[0.75, 0.25])),
            "dependants": int(rng.choice([0, 1, 2, 3, 4], p=[0.25, 0.25, 0.25, 0.15, 0.10])),
            "loss_tolerance": str(rng.choice(["Low", "Moderate", "High"], p=[0.25, 0.50, 0.25])),
            "high_interest_debt": bool(rng.choice([True, False], p=[0.15, 0.85])),
            "emergency_fund_adequate": bool(rng.choice([True, False], p=[0.65, 0.35])),
            "initial_amount": 100000
        })
    return population

population = generate_population()
population_records = []
constraint_groups = {"Debt": [], "Emergency reserve": [], "Near-term goal": []}
changed_count = 0

for i, client in enumerate(population):
    result = advise(client)
    changed = any(abs(result["recommended"][a] - result["unconstrained"][a]) > 1e-9 for a in ASSETS)
    changed_count += int(changed)
    paths = simulate_asset_returns(client["horizon"], n_paths=3000, seed=8000+i)

    # Fifth percentile under score-only and constrained recommendations.
    p5_unconstrained = np.percentile(
        simulate_allocation(client["initial_amount"], result["unconstrained"], paths), 5
    )
    p5_recommended = np.percentile(
        simulate_allocation(client["initial_amount"], result["recommended"], paths), 5
    )

    population_records.append({
        "Client": client["name"], "Score": result["score"], "Band": result["band"],
        "Changed": changed, "Constraints": ", ".join(result["constraints_fired"]) or "None",
        "P5 score-only (₹)": p5_unconstrained, "P5 constrained (₹)": p5_recommended
    })

    for constraint in result["constraints_fired"]:
        constraint_groups[constraint].append((p5_unconstrained, p5_recommended))

population_df = pd.DataFrame(population_records)
print(f"Recommendations changed: {changed_count} / {len(population)} "
      f"({changed_count/len(population):.1%})")
display(population_df.head(10).style.format({
    "P5 score-only (₹)": "₹{:,.0f}", "P5 constrained (₹)": "₹{:,.0f}"
}))

impact_rows = []
for constraint, outcomes in constraint_groups.items():
    if outcomes:
        arr = np.array(outcomes)
        impact_rows.append({
            "Constraint": constraint, "Clients affected": len(outcomes),
            "Mean P5 score-only (₹)": arr[:, 0].mean(),
            "Mean P5 constrained (₹)": arr[:, 1].mean(),
            "Mean change in P5 (₹)": (arr[:, 1] - arr[:, 0]).mean()
        })
impact_table = pd.DataFrame(impact_rows)
display(impact_table.style.format({
    "Mean P5 score-only (₹)": "₹{:,.0f}",
    "Mean P5 constrained (₹)": "₹{:,.0f}",
    "Mean change in P5 (₹)": "₹{:,.0f}"
}))

### Exercise 1 — Interpretation

- The first output gives the number and percentage of generated clients whose asset allocation changes after hard constraints.
- The constraint-impact table reports, for clients affected by each rule, the mean fifth-percentile final value under both versions. The mean difference is descriptive; it does not prove that a constraint is universally beneficial.
- The **debt constraint** is expected to make the largest structural change because it replaces the entire score-based portfolio with the debt-repayment priority. The actual ranking by number of affected clients and mean fifth-percentile difference is computed by the cell above.
- **Can any constraint be folded into the score?** No, not without changing its meaning. A weighted score is compensatory: a high value on one attribute can offset a low value on another. The lab's debt, emergency reserve, and near-term goal rules are intended as non-compensatory conditions. They should remain a separate constraint layer.

## 7. Exercise 2 — Band boundaries and smooth interpolation

The banded system can produce a discontinuous allocation change when a score crosses a boundary. First, inspect the score immediately below and above every boundary. Then replace the bands with linear interpolation between the band-centre allocations (centres 12, 37, 62 and 87). Outside the first and last centres, the nearest centre allocation is retained.

In [ ]:
def banded_equity(score):
    return score_band(score)[1]["Equity"]

boundary_rows = []
for boundary in [24, 49, 74]:
    boundary_rows.append({
        "Boundary": boundary,
        "Score just below": boundary,
        "Equity below (%)": 100 * banded_equity(boundary),
        "Score just above": boundary + 1,
        "Equity above (%)": 100 * banded_equity(boundary + 1),
        "Jump from one point (%)": 100 * (banded_equity(boundary + 1) - banded_equity(boundary))
    })
boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)
largest_banded_jump = boundary_table["Jump from one point (%)"].abs().max()
print(f"Largest one-point equity change in banded system: {largest_banded_jump:.1f} percentage points")

CENTRES = np.array([12, 37, 62, 87], dtype=float)
CENTRE_EQUITY = np.array([0.20, 0.40, 0.60, 0.80])

def smooth_equity(score):
    return float(np.interp(score, CENTRES, CENTRE_EQUITY))

def smooth_allocation(score):
    equity = smooth_equity(score)
    # Keep a simple 3:1 split of the non-equity portion between bonds and cash.
    remaining = 1.0 - equity
    return {"Equity": equity, "Bonds": remaining * 0.75, "Cash": remaining * 0.25}

scores = np.arange(0, 101)
smooth_changes = np.diff([smooth_equity(s) for s in scores]) * 100
print(f"Largest one-point equity change in smooth system: {np.max(np.abs(smooth_changes)):.2f} percentage points")

boundary_compare = []
for boundary in [24, 49, 74]:
    boundary_compare.append({
        "Boundary": boundary,
        "Banded equity at boundary (%)": 100 * banded_equity(boundary),
        "Banded equity next point (%)": 100 * banded_equity(boundary + 1),
        "Smooth equity at boundary (%)": 100 * smooth_equity(boundary),
        "Smooth equity next point (%)": 100 * smooth_equity(boundary + 1)
    })
display(pd.DataFrame(boundary_compare))

plt.figure(figsize=(10, 5))
plt.step(scores, [100*banded_equity(s) for s in scores], where="post", label="Banded allocation")
plt.plot(scores, [100*smooth_equity(s) for s in scores], label="Smooth interpolation")
plt.xlabel("Risk score")
plt.ylabel("Equity allocation (%)")
plt.title("Equity allocation: banded vs smooth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### 7.1 Ten-year comparison for a client at the boundary
Use score 49 (the boundary immediately before the Balanced band) as the example. Compare the banded allocation at score 49 with the smooth allocation at score 49 over ten years, using identical simulated asset-return paths.

In [ ]:
boundary_score = 49
boundary_client_amount = 100000
boundary_paths = simulate_asset_returns(10, n_paths=20000, seed=9912)

band_alloc = score_band(boundary_score)[1]
smooth_alloc = smooth_allocation(boundary_score)

band_values = simulate_allocation(boundary_client_amount, band_alloc, boundary_paths)
smooth_values = simulate_allocation(boundary_client_amount, smooth_alloc, boundary_paths)

boundary_ten_year = pd.DataFrame([
    {"Method": "Banded", "Allocation": allocation_string(band_alloc),
     **summarize_outcomes(band_values, boundary_client_amount)},
    {"Method": "Smooth", "Allocation": allocation_string(smooth_alloc),
     **summarize_outcomes(smooth_values, boundary_client_amount)}
])
display(boundary_ten_year.style.format({
    "Median final value (₹)": "₹{:,.0f}",
    "5th percentile (₹)": "₹{:,.0f}",
    "Probability of loss (%)": "{:.2f}%"
}))

print("Banded equity at score 49:", f"{band_alloc['Equity']:.1%}")
print("Smooth equity at score 49:", f"{smooth_alloc['Equity']:.1%}")

### Exercise 2 — Interpretation

- In the banded system, a one-point score increase at a boundary can cause a large step in equity allocation. The table reports each boundary's exact jump, and the code calculates the largest jump.
- Smooth interpolation removes abrupt jumps between band centres. With the selected centres, the equity allocation changes gradually rather than moving directly from one fixed band to another.
- For the ten-year boundary example, compare the median, fifth percentile, and probability of loss in the generated table. A higher median alone is not enough to declare a method superior; the downside percentile and loss probability must also be considered.
- Smooth interpolation is less sensitive to an arbitrary one-point threshold crossing, but it does not remove the need for hard constraints. The same debt, reserve, and short-horizon rules should still override the interpolated score-based allocation.

## 8. Conclusion

This experiment implements a two-layer advisory expert system. The scoring layer translates client attributes into a standard allocation and the constraint layer applies non-compensatory priorities with an explicit explanation trace. Monte Carlo simulation evaluates the distribution of outcomes, including the median, fifth percentile, and probability of loss. The population experiment illustrates how often constraints alter advice, while the boundary experiment demonstrates the discontinuity of fixed score bands and the smoother behaviour of interpolation.

**Limitations:** Normal, independent annual returns are a simplifying assumption. The model omits inflation, taxes, fees, changing asset correlations, fat tails, market regimes, rebalancing costs, contributions/withdrawals, and the client's full balance sheet. The simulation is therefore an educational comparison tool, not a forecast or a guarantee.